In [ ]:
# Predictive Analytics: Kundenabwanderungs-Vorhersage (Telco Churn)
**Ein wirtschaftsmathematischer Machine-Learning-Ansatz kombiniert mit Business-Coaching-Strategien zur Kundenrückgewinnung.**

## 1. Projektübersicht & Business-Case
Unternehmen verlieren jährlich Milliarden durch Kundenabwanderung (Churn). In diesem Projekt nutzen wir ein reales Kundendatenset eines Telekommunikationskonzerns, um unzufriedene Kunden rechtzeitig zu identifizieren, bevor sie kündigen. 

Als **Business Coach** und Analyst reicht es nicht, nur Code zu schreiben. Das Ziel ist es, die stärksten Treiber der Abwanderung zu isolieren (z.B. Produktmängel oder Vertragsstrukturen) und das Machine-Learning-Modell betriebswirtschaftlich so zu optimieren, dass der maximale Umsatz gerettet wird.

## 2. Mathematisches Fundament (Logistische Regression)
Da es sich um eine binäre Klassifikation handelt (Abwanderung: Ja=1 / Nein=0), transformieren wir die Wahrscheinlichkeit $p$ über die **Odds** ($p / (1-p)$) und die **Log-Odds** (Logit-Funktion) in den unendlichen Wertebereich ($-\infty$ bis $+\infty$), um sie mit der Regressionsgeraden gleichzusetzen:

$$\ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 \cdot x_1 + \dots + \beta_n \cdot x_n = z$$

Durch Auflösen nach $p$ erhalten wir die **Sigmoid-Funktion**, die jede Vorhersage sauber zwischen 0 und 1 abbildet:
$$p = \frac{1}{1 + e^{-z}}$$

Die Optimierung der Gewichte ($\beta$) erfolgt iterativ über das **Gradientenverfahren (Gradient Descent)** zur Minimierung der konvexen Kostenfunktion (**Log-Loss**).


In [1]:
import pandas as pd

df = pd.read_csv('Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# Absolute und relative Häufigkeiten der Abwanderung
print(df['Churn'].value_counts())
print("\nProzentuale Verteilung:")
print(df['Churn'].value_counts(normalize=True) * 100)

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Prozentuale Verteilung:
Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [7]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [9]:
# 1. Leerzeichen durch NaN (Not a Number) ersetzen und in Float konvertieren
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# 2. Prüfen, wie viele fehlende Werte dadurch entstanden sind
print("Fehlende Werte in TotalCharges:", df['TotalCharges'].isnull().sum())

# 3. Da es bei über 7000 Zeilen nur 11 Neukunden betrifft, löschen wir diese Zeilen direkt
df = df.dropna(subset=['TotalCharges'])

# 4. Kontrolle des Datentyps
print("\nNeuer Datentyp von TotalCharges:", df['TotalCharges'].dtype)


Fehlende Werte in TotalCharges: 11

Neuer Datentyp von TotalCharges: float64


In [11]:
# Kreuztabelle: Vertragsart vs. Churn (in Prozent)
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
print(contract_churn)


Churn                  No        Yes
Contract                            
Month-to-month  57.290323  42.709677
One year        88.722826  11.277174
Two year        97.151335   2.848665


In [13]:
# 1. Die Zielvariable 'Churn' in 0 und 1 umwandeln
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

# 2. Ausgewählte kategorische Variablen in Dummy-Variablen (0/1) umwandeln
# Wir fokussieren uns für den Start auf die wichtigsten Hebel: Contract und InternetService
features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Contract', 'InternetService']
X = pd.get_dummies(df[features], drop_first=True)
y = df['Churn']

# 3. Kontrollblick auf die mathematische Matrix
print("Die ersten Zeilen unserer Feature-Matrix X:")
print(X.head())


Die ersten Zeilen unserer Feature-Matrix X:
   tenure  MonthlyCharges  TotalCharges  Contract_One year  Contract_Two year  \
0       1           29.85         29.85              False              False   
1      34           56.95       1889.50               True              False   
2       2           53.85        108.15              False              False   
3      45           42.30       1840.75               True              False   
4       2           70.70        151.65              False              False   

   InternetService_Fiber optic  InternetService_No  
0                        False               False  
1                        False               False  
2                        False               False  
3                        False               False  
4                         True               False  


In [17]:
from sklearn.model_selection import train_test_split

# Aufteilung in 80% Training und 20% Test mit fixiertem Zufall
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Kontrolle der Dimensionen (Matrizen-Größen)
print("Größe der Trainings-Matrix X_train:", X_train.shape)
print("Größe der Test-Matrix X_test:", X_test.shape)


Größe der Trainings-Matrix X_train: (5625, 7)
Größe der Test-Matrix X_test: (1407, 7)


In [19]:
from sklearn.linear_model import LogisticRegression

# 1. Das Modell initialisieren (wir erhöhen max_iter, damit der Algorithmus genug Zeit zum Konvergieren hat)
model = LogisticRegression(max_iter=1000)

# 2. Das Training starten (Hier berechnet Python das Gradientenverfahren!)
model.fit(X_train, y_train)

print("Das Modell wurde erfolgreich auf den Trainingsdaten trainiert!")


Das Modell wurde erfolgreich auf den Trainingsdaten trainiert!


In [21]:
# Die berechneten Beta-Gewichte anzeigen
coef_df = pd.DataFrame({'Feature': X.columns, 'Beta (Gewicht)': model.coef_[0]})
print("Der berechnete Achsenabschnitt (Beta_0):", model.intercept_[0])
print("\nDie Gewichte der einzelnen Features:")
print(coef_df.sort_values(by='Beta (Gewicht)', ascending=False))


Der berechnete Achsenabschnitt (Beta_0): -0.04337382329023673

Die Gewichte der einzelnen Features:
                       Feature  Beta (Gewicht)
5  InternetService_Fiber optic        1.077380
2                 TotalCharges        0.000349
1               MonthlyCharges       -0.002321
0                       tenure       -0.060949
3            Contract_One year       -0.953690
6           InternetService_No       -0.991204
4            Contract_Two year       -1.707810


In [23]:
# 1. Wahrscheinlichkeiten für die Testdaten berechnen
y_pred_proba = model.predict_proba(X_test)[:, 1]

# 2. Die konkrete Klassifikation (0 oder 1) bei einem Standard-Schwellenwert von 0.5
y_pred = model.predict(X_test)

# Einen kleinen DataFrame zur Visualisierung der Vorhersagen bauen
predictions_df = pd.DataFrame({
    'Echte Kündigung (y_test)': y_test.values,
    'Vorhergesagte Wahrscheinlichkeit': y_pred_proba,
    'KI-Entscheidung (y_pred)': y_pred
})
print(predictions_df.head(10))


   Echte Kündigung (y_test)  Vorhergesagte Wahrscheinlichkeit  \
0                         0                          0.002489   
1                         0                          0.110352   
2                         1                          0.617561   
3                         0                          0.152304   
4                         0                          0.396275   
5                         0                          0.434772   
6                         0                          0.050762   
7                         0                          0.597894   
8                         0                          0.238602   
9                         0                          0.017355   

   KI-Entscheidung (y_pred)  
0                         0  
1                         0  
2                         1  
3                         0  
4                         0  
5                         0  
6                         0  
7                         1  
8             

In [25]:
from sklearn.metrics import confusion_matrix, classification_report

# 1. Konfusionsmatrix berechnen
cm = confusion_matrix(y_test, y_pred)
print("Konfusionsmatrix:")
print(cm)

# 2. Der detaillierte mathematische Report
print("\nKlassifikations-Bericht:")
print(classification_report(y_test, y_pred))


Konfusionsmatrix:
[[926 107]
 [196 178]]

Klassifikations-Bericht:
              precision    recall  f1-score   support

           0       0.83      0.90      0.86      1033
           1       0.62      0.48      0.54       374

    accuracy                           0.78      1407
   macro avg       0.72      0.69      0.70      1407
weighted avg       0.77      0.78      0.77      1407



In [27]:
# Strategische Absenkung des Schwellenwerts auf 30% (0.3)
# Wir wollen mehr Risiko-Kunden einfangen!
y_pred_coaching = (y_pred_proba >= 0.3).astype(int)

# Neuer Klassifikations-Bericht
print("Optimierter Klassifikations-Bericht (Threshold = 0.3):")
print(classification_report(y_test, y_pred_coaching))


Optimierter Klassifikations-Bericht (Threshold = 0.3):
              precision    recall  f1-score   support

           0       0.90      0.72      0.80      1033
           1       0.50      0.78      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.75      1407

